# VieNeu-TTS-v2 × vLLM-Omni (Colab T4)

Serve **`pnnbao-ump/VieNeu-TTS-v2`** natively inside **vLLM-Omni** as a first-class TTS architecture — no FastAPI wrapper, no monkey-patches, no edits to the checkpoint's `config.json`.

Target command (the one this notebook runs end-to-end):

```bash
vllm serve pnnbao-ump/VieNeu-TTS-v2 --omni --port 8000
```

Then call the OpenAI-compatible endpoint:

```python
OpenAI(base_url="http://localhost:8000/v1").audio.speech.create(
    model="pnnbao-ump/VieNeu-TTS-v2", input="Xin chào"
)
```

**Fork + branch:** `justHman/vllm-omni@feat/vieneu-tts` — branched from upstream tag `v0.19.0rc1` (the only release compatible with `vllm==0.19.0`, which is what this Colab T4 runtime uses).

**What this notebook does**
1. Detect Colab runtime (assert GPU / CUDA).
2. Mount Google Drive (optional, forHF cache persistence).
3. Install pinned `vllm==0.19.0`, then the fork with the new `vieneu` extra (`sea-g2p`, `neucodec`).
4. Pre-fetch the checkpoint `pnnbao-ump/VieNeu-TTS-v2` and the external NeuCodec repo `neuphonic/neucodec`.
5. Launch `vllm serve ... --omni` in the background.
6. Wait for `/v1/models` to come up, then POST to `/v1/audio/speech` and save a `.wav`.
7. Play the audio inline.

> Note: this notebook is the **first end-to-end runtime test** of the integration. If start-up or inference fails, the failure output is captured in the launch cell + `/tmp/vllm_serve.log` — paste it back so the model/serving wiring can be fixed.

In [ ]:
# ============================================================
# 0. Preflight — environment detection (fail fast)
# ============================================================
import os, sys, subprocess, time, json, textwrap, urllib.request

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass

print("\u2713 Colab:", IN_COLAB)
print("\u2713 Python:", sys.version.split()[0])

# Assert CUDA GPU (T4 expected on free Colab).
try:
    import torch
    print("\u2713 torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("\u2713 GPU:", torch.cuda.get_device_name(0))
        print("\u2713 CUDA:", torch.version.cuda)
    else:
        raise RuntimeError("No CUDA GPU detected. Colab T4 required: Runtime > Change runtime type > T4 GPU.")
except ImportError:
    raise RuntimeError("torch not in the Colab image; this notebook expects the default GPU image.")

PORT = 8000
MODEL = "pnnbao-ump/VieNeu-TTS-v2"
FORK_REPO = "https://github.com/justHman/vllm-omni.git"
FORK_BRANCH = "feat/vieneu-tts"
print("\u2713 model:", MODEL)
print("\u2713 fork:", FORK_REPO, "@", FORK_BRANCH)

In [ ]:
# ============================================================
# 1. (Optional) Mount Drive for HF cache persistence across sessions
# ============================================================
MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        MOUNTED = True
        print("\u2713 Drive mounted")
    except Exception as e:
        print("\u26a0 Drive mount skipped:", e)

# Point HF cache at Drive so the ~600MB checkpoint + NeuCodec don't re-download every session.
if MOUNTED:
    hf_cache = "/content/drive/MyDrive/hf_cache"
    os.makedirs(hf_cache, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache
    os.environ["TRANSFORMERS_CACHE"] = os.path.join(hf_cache, "transformers")
    os.environ["HF_HUB_CACHE"] = os.path.join(hf_cache, "hub")
    print("\u2713 HF_HOME =", hf_cache)
else:
    print("\u26a0 HF cache stays in-session (/root/.cache/huggingface).")

In [ ]:
# ============================================================
# 2. Install vllm==0.19.0 (the only release vllm-omni rc1 is compatible with)
# ============================================================
# `uv pip install --system` is ~10x faster than pip. --torch-backend=auto makes
# uv resolve torch/torchaudio/torchvision from the CUDA index matching this GPU
# (T4 -> cu13x), which also prevents the torchaudio ABI mismatch that breaks
# `neucodec` import (libtorchaudio.so) on a plain pip install.
!pip install -q uv
!uv pip install --system vllm==0.19.0 --torch-backend=auto
import vllm
print("✓ vllm:", vllm.__version__)
assert vllm.__version__.startswith("0.19.0"), f"expected vllm 0.19.0, got {vllm.__version__}"


In [ ]:
# ============================================================
# 3. Clone the fork + install vllm-omni with the new `vieneu` extra
# ============================================================
!rm -rf /content/vllm-omni
!git clone --depth 1 --branch {FORK_BRANCH} {FORK_REPO} /content/vllm-omni
!cd /content/vllm-omni && git log --oneline -1

# Install the fork editable + the vieneu extra (adds sea-g2p + neucodec).
import os, importlib, subprocess, sys
os.environ["VLLM_OMNI_TARGET_DEVICE"] = "cuda"
!uv pip install --system -e /content/vllm-omni
!uv pip install --system -e "/content/vllm-omni[vieneu]" --torch-backend=auto

# Belt-and-suspenders: pin torchaudio to the exact active torch build.
# neucodec imports torchaudio eagerly and crashes on an ABI-mismatched
# libtorchaudio.so. uv --torch-backend=auto aligns torch-family wheels, but if
# neucodec doesn't declare a torchaudio dep, Colab's preinstalled (ABI-broken)
# one survives -- this pin is the final authority.
import torch
_torch_version = torch.__version__.split("+")[0]
_torch_cuda = (torch.version.cuda or "").replace(".", "")
_torch_index = f"https://download.pytorch.org/whl/cu{_torch_cuda}" if _torch_cuda else None
print("Pinning torchaudio ==", _torch_version, "from", _torch_index or "PyPI")
_ta_cmd = [sys.executable, "-m", "uv", "pip", "install", "--system",
           "--force-reinstall", "--no-deps", f"torchaudio=={_torch_version}"]
if _torch_index:
    _ta_cmd += ["--index-url", _torch_index]
subprocess.check_call(_ta_cmd)

# Sanity: confirm the extra deps import.
for mod in ["sea_g2p", "torchaudio", "neucodec"]:
    try:
        importlib.import_module(mod)
        print(f"✓ {mod} importable")
    except Exception as e:
        print(f"✗ {mod} import FAILED:", e)
        raise


In [ ]:
# ============================================================
# 4. Pre-download the checkpoint + external NeuCodec (avoids first-serve timeout)
# ============================================================
from huggingface_hub import snapshot_download

print("Downloading", MODEL, "...")
mp = snapshot_download(MODEL, allow_patterns=["*.json", "*.txt", "*.safetensors", "*.model", "voices.json", "tokenizer*"])
print("\u2713 checkpoint at:", mp)

print("Downloading neuphonic/neucodec (external NeuCodec) ...")
npp = snapshot_download("neuphonic/neucodec")
print("\u2713 neucodec at:", npp)

In [ ]:
# ============================================================
# 5. Launch `vllm serve ... --omni` in the background
# ============================================================
import subprocess, os, time, signal

LOG = "/tmp/vllm_serve.log"
# Kill any stale server on PORT.
!fuser -k {PORT}/tcp 2>/dev/null || true

cmd = [
    "vllm", "serve", MODEL,
    "--omni",
    "--port", str(PORT),
    "--host", "0.0.0.0",
    "--trust-remote-code",
]
print("Launching:", " ".join(cmd))
print("Logs ->", LOG)

logf = open(LOG, "w", buffering=1)
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd="/content/vllm-omni", env=dict(os.environ))
print("\u2713 server PID:", proc.pid)

In [ ]:
# ============================================================
# 6. Wait for /v1/models to come up (print log tail if it times out)
# ============================================================
import urllib.request, json, time

def get_models(timeout=1200):
    url = f"http://localhost:{PORT}/v1/models"
    start = time.time()
    last_err = None
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    return json.loads(r.read())
        except Exception as e:
            last_err = e
        if proc.poll() is not None:
            raise RuntimeError(f"server process exited early (code {proc.returncode}). See {LOG}.")
        time.sleep(5)
    raise RuntimeError(f"timed out waiting for {url}. last_err={last_err}. Tail of {LOG}:\n" + tail(LOG))

def tail(path, n=80):
    try:
        with open(path, errors="replace") as f:
            lines = f.readlines()
        return "".join(lines[-n:])
    except Exception as e:
        return f"<could not read {path}: {e}>"

print("Waiting for server (up to 20 min for first-time compile) ...")
models = get_models(timeout=1200)
print("\u2713 /v1/models:")
print(json.dumps(models, indent=2))

In [ ]:
# ============================================================
# 7. POST /v1/audio/speech (preset voice) and save a WAV
# ============================================================
import requests, base64, struct, numpy as np

def speech(text="Xin ch\u00e0o, \u0111\u00e2y l\u00e0 gi\u1ecdng n\u00f3i t\u1eeb VieNeu-TTS.", voice=None, response_format="wav", stream=False):
    payload = {"model": MODEL, "input": text, "response_format": response_format}
    if voice:
        payload["voice"] = voice
    if stream:
        payload["stream"] = True
    r = requests.post(f"http://localhost:{PORT}/v1/audio/speech", json=payload, stream=stream, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"speech failed: {r.status_code} {r.text[:2000]}")
    return r

# Default preset voice = "Ly" (one of the 7 v2 presets in voices.json).
r = speech(voice="Ly", response_format="wav", stream=False)
out_wav = "/content/vieneu_demo.wav"
with open(out_wav, "wb") as f:
    f.write(r.content)
print("\u2713 wrote", out_wav, len(r.content), "bytes")

In [ ]:
# ============================================================
# 8. Play the generated audio inline
# ============================================================
from IPython.display import Audio, display, HTML
print("VieNeu-TTS-v2 /v1/audio/speech output:")
display(Audio(out_wav, autoplay=False))

In [ ]:
# ============================================================
# 9. (Diagnostic) Show last 60 log lines — paste this back if anything failed
# ============================================================
print(tail(LOG, n=60))

In [ ]:
# ============================================================
# 10. (Optional) Voice cloning from ref_audio + ref_text
# ============================================================
# Upload a 1-30s Vietnamese/English clip + its transcript, then:
#
# from google.colab import files
# up = files.upload()
# ref_path = list(up.keys())[0]
# import base64, requests
# with open(ref_path,"rb") as f: b64 = base64.b64encode(f.read()).decode()
# r = requests.post(f"http://localhost:{PORT}/v1/audio/speech", json={
#     "model": MODEL, "input": "H\u00e3y n\u00f3i b\u1eb1ng gi\u1ecdng c\u1ee7a t\u00f4i.",
#     "ref_audio": f"data:audio/wav;base64,{b64}",
#     "ref_text": "<exact transcript of the clip>",
# }, timeout=180)
# open("/content/vieneu_clone.wav","wb").write(r.content)

## Notes / known limitations

- **Cells 3-4 use `uv pip install --system --torch-backend=auto`** for ~10x speed over pip, and so uv resolves torch/torchaudio/torchvision from the CUDA index matching this GPU -- preventing the `libtorchaudio.so` ABI mismatch that breaks `neucodec`'s eager torchaudio import. A final `torchaudio==<torch>` pin in cell 4 is kept as belt-and-suspenders in case `neucodec` doesn't declare a torchaudio dep.
- **First launch** downloads + JIT-compiles a few CUDA kernels; the wait cell allows up to 20 min. The first `import transformers` inside the `vllm` CLI is slow (~30-60s) as it reads every model file -- do NOT interrupt it (a KeyboardInterrupt during that import looks like a crash but isn't). Subsequent restarts are faster if Drive cache is mounted.
- **Preset voices** (`Ly` default) come from the checkpoint's `voices.json` -- 7 voices for v2, licensed **CC BY-NC 4.0** (non-commercial), separate from the Apache-2.0 model weights.
- **`ref_audio` + `ref_text` cloning** is wired at the serving layer via `encode_ref_audio` (NeuCodec encode API verified against `neuphonic/neucodec` + `pnnbao97/VieNeu-TTS` repo). See cell 10.
- If `/v1/models` never comes up, paste cell 9 (log tail) back -- most likely a missing dep, a CUDA/cu13 mismatch on the image, or an unhandled shape in the codec stage. The log will say which.
